In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Donnée principal
df = pd.read_csv("./data/Processed.csv")
df.head()

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ2,PHQ3,PHQ4,PHQ5,PHQ6,PHQ7,PHQ8,PHQ9,Depression Value,Depression Label
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,3,4,3,...,2,3,2,2,2,2,3,2,20,Severe Depression
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,3,4,...,2,2,2,2,2,2,2,2,19,Moderately Severe Depression
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,...,0,0,0,0,0,0,0,0,0,No Depression
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,1,2,...,1,2,1,2,1,2,2,1,14,Moderate Depression
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,4,4,...,3,3,3,1,3,0,3,3,20,Severe Depression


# Remarque
Pour le facteur économique de chaque etudiant, Le 'Processed.csv' ne contient que le bourse, alors, j'ai choisit de le completer avec un autre donnée provenant de UCI Machine learning , nomé data.csv

In [2]:
#Donnée complementaire
df_data = pd.read_csv('./data/second_data/data.csv', sep=";")
df_data.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


# Avant de le fusionner:
Il faut regarder d'abord si les deux datasets contiennent des valeurs manquantes

In [3]:
#Pour le donnée principal
df.isna().sum()

Age                      0
Gender                   0
University               0
Department               0
Academic_Year            0
Current_CGPA             0
waiver_or_scholarship    0
PSS1                     0
PSS2                     0
PSS3                     0
PSS4                     0
PSS5                     0
PSS6                     0
PSS7                     0
PSS8                     0
PSS9                     0
PSS10                    0
Stress Value             0
Stress Label             0
GAD1                     0
GAD2                     0
GAD3                     0
GAD4                     0
GAD5                     0
GAD6                     0
GAD7                     0
Anxiety Value            0
Anxiety Label            0
PHQ1                     0
PHQ2                     0
PHQ3                     0
PHQ4                     0
PHQ5                     0
PHQ6                     0
PHQ7                     0
PHQ8                     0
PHQ9                     0
D

In [4]:
# Pour le donnée complementaire
df_data.isna().sum()

Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance\t                      0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship holder                                0
Age at enrol

In [5]:
#taille du Processed.csv
df.shape

(2028, 39)

In [6]:
#taille du data.csv
df_data.shape

(4424, 37)

In [7]:
#remplace les valeurs numérique en valeur categoriel pour que le type de donnée des bourses des deux données sont identiques
df_data["scholarship_flag"] = df_data["Scholarship holder"].map({1: "Yes", 0: "No"})
df["scholarship_flag"] = df["waiver_or_scholarship"].map(
    {"Yes": "Yes", "No": "No"})

In [8]:
#Considerer les autres valeurs(apart yes/no) par no
df["scholarship_flag"] = df["scholarship_flag"].fillna("No")

In [9]:
#Classifie l'emploi des parents
def categorize_occupation(code):
    try:
        code = int(code)
    except (ValueError, TypeError):
        return "Non_specifie"
    if code >= 90:
        return "Non_specifie_ou_sans_emploi"
    first_digit = code // 10 if code >= 10 else code
    if first_digit <= 3:
        return "Cadre_intellectuel_superieur"
    elif first_digit <= 5:
        return "Employe_technicien"
    else:
        return "Ouvrier_metier_manuel"

df_data["Parental_SES_donor"] = df_data.apply(
    lambda r: categorize_occupation(
        min(r["Mother's occupation"], r["Father's occupation"])
    ),
    axis=1,
)

In [10]:
#On stocke les variables à completer dans un variable
df_data_cols = ["Debtor", "Tuition fees up to date", "Parental_SES_donor"]

In [11]:
enriched_rows = []
for grp_value, sub_df in df.groupby("scholarship_flag"):
    sub_df = df_data[df_data["scholarship_flag"] == grp_value]
    if sub_df.empty:
        sub_df = df  # fallback de securite (ne devrait pas arriver ici)

    sampled_idx = rng.choice(sub_df.index, size=len(sub_df), replace=True)
    sampled = sub_df.loc[sampled_idx, df_data_cols].reset_index(drop=True)
    sampled.index = sub_df.index
    enriched_rows.append(sampled)

enriched = pd.concat(enriched_rows).sort_index()

df["Financial_Debtor_proxy"] = enriched["Debtor"].map({1: "Yes", 0: "No"})
df["Tuition_UpToDate_proxy"] = enriched["Tuition fees up to date"].map(
    {1: "Yes", 0: "No"}
)
df["Parental_SES_proxy"] = enriched["Parental_SES_donor"]

In [12]:
#donne du score de vulnérabilité économique de chaque etudiant
def vulnerability_score(row):
    score = 0
    if row["scholarship_flag"] == "No":
        score += 1
    if row["Financial_Debtor_proxy"] == "Yes":
        score += 1
    if row["Tuition_UpToDate_proxy"] == "No":
        score += 1
    if row["Parental_SES_proxy"] in (
        "Ouvrier_metier_manuel",
        "Non_specifie_ou_sans_emploi",
    ):
        score += 1
    return score

In [13]:
df["Economic_Vulnerability_Index"] = df.apply(vulnerability_score, axis=1)

In [14]:
# Nettoyage colonne technique
df = df.drop(columns=["scholarship_flag"])

In [15]:
df

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ6,PHQ7,PHQ8,PHQ9,Depression Value,Depression Label,Financial_Debtor_proxy,Tuition_UpToDate_proxy,Parental_SES_proxy,Economic_Vulnerability_Index
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,3,4,3,...,2,2,3,2,20,Severe Depression,No,Yes,Cadre_intellectuel_superieur,1
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,3,4,...,2,2,2,2,19,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,...,0,0,0,0,0,No Depression,Yes,No,Employe_technicien,3
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,1,2,...,1,2,2,1,14,Moderate Depression,No,Yes,Employe_technicien,1
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,4,4,...,3,0,3,3,20,Severe Depression,No,Yes,Employe_technicien,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,18-22,Male,Dhaka University (DU),Other,Second Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,2,2,3,18,Moderately Severe Depression,No,Yes,Employe_technicien,1
2024,18-22,Female,Bangladesh Agricultural University (BAU),Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,1,1,3,19,Moderately Severe Depression,No,Yes,Employe_technicien,1
2025,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,2,1,...,2,2,2,2,15,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2026,23-26,Female,Dhaka University (DU),Other,Third Year or Equivalent,3.40 - 3.79,No,4,4,4,...,3,3,0,0,21,Severe Depression,No,Yes,Employe_technicien,1


In [16]:
print("\n--- Verification : distribution conditionnelle preservee ---")
print(
    pd.crosstab(
        df["waiver_or_scholarship"], df["Financial_Debtor_proxy"], normalize="index"
    ).round(3)
)
print(
    pd.crosstab(
        df["waiver_or_scholarship"], df["Parental_SES_proxy"], normalize="index"
    ).round(3)
)



--- Verification : distribution conditionnelle preservee ---
Financial_Debtor_proxy     No    Yes
waiver_or_scholarship               
No                      0.875  0.125
Yes                     0.899  0.101
Parental_SES_proxy     Cadre_intellectuel_superieur  Employe_technicien  \
waiver_or_scholarship                                                     
No                                            0.283               0.305   
Yes                                           0.303               0.279   

Parental_SES_proxy     Non_specifie_ou_sans_emploi  Ouvrier_metier_manuel  
waiver_or_scholarship                                                      
No                                           0.039                  0.373  
Yes                                          0.028                  0.390  


In [17]:
output_path = "./data/data_cleaned/Processed_enriched.csv"
df.to_csv(output_path, index=False)
print(f"\nFichier enrichi sauvegarde: {output_path}")
print(f"Nouvelles colonnes ajoutees: Financial_Debtor_proxy, "
      f"Tuition_UpToDate_proxy, Parental_SES_proxy, Economic_Vulnerability_Index")



Fichier enrichi sauvegarde: ./data/data_cleaned/Processed_enriched.csv
Nouvelles colonnes ajoutees: Financial_Debtor_proxy, Tuition_UpToDate_proxy, Parental_SES_proxy, Economic_Vulnerability_Index


In [18]:
df

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ6,PHQ7,PHQ8,PHQ9,Depression Value,Depression Label,Financial_Debtor_proxy,Tuition_UpToDate_proxy,Parental_SES_proxy,Economic_Vulnerability_Index
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,3,4,3,...,2,2,3,2,20,Severe Depression,No,Yes,Cadre_intellectuel_superieur,1
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,3,4,...,2,2,2,2,19,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,...,0,0,0,0,0,No Depression,Yes,No,Employe_technicien,3
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,1,2,...,1,2,2,1,14,Moderate Depression,No,Yes,Employe_technicien,1
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,4,4,...,3,0,3,3,20,Severe Depression,No,Yes,Employe_technicien,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,18-22,Male,Dhaka University (DU),Other,Second Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,2,2,3,18,Moderately Severe Depression,No,Yes,Employe_technicien,1
2024,18-22,Female,Bangladesh Agricultural University (BAU),Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,1,1,3,19,Moderately Severe Depression,No,Yes,Employe_technicien,1
2025,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,2,1,...,2,2,2,2,15,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2026,23-26,Female,Dhaka University (DU),Other,Third Year or Equivalent,3.40 - 3.79,No,4,4,4,...,3,3,0,0,21,Severe Depression,No,Yes,Employe_technicien,1


In [19]:
# Type de donné
df.dtypes

Age                             object
Gender                          object
University                      object
Department                      object
Academic_Year                   object
Current_CGPA                    object
waiver_or_scholarship           object
PSS1                             int64
PSS2                             int64
PSS3                             int64
PSS4                             int64
PSS5                             int64
PSS6                             int64
PSS7                             int64
PSS8                             int64
PSS9                             int64
PSS10                            int64
Stress Value                     int64
Stress Label                    object
GAD1                             int64
GAD2                             int64
GAD3                             int64
GAD4                             int64
GAD5                             int64
GAD6                             int64
GAD7                     

# Construction du variable cible
Bien que le donnée est alors rempli, on va construire la variable cible pour l'abondant universitaire

# les principales facteurs de l'abondant à partir de ce donnée
- Académique (35%)
- Santé Mentale (35%)
- Economique (30%)

## Normalisation manuel:
On fait un normalisation manuel, c'est à dire, on le mormalise avec un nombre entre 0 et 1 à partir de l'intensité de chaque variable

In [20]:
#Pilier académique
df["Current_CGPA"].value_counts()

Current_CGPA
3.00 - 3.39    583
3.40 - 3.79    560
2.50 - 2.99    389
3.80 - 4.00    241
Other          171
Below 2.50      84
Name: count, dtype: int64

In [21]:
cgpa_map = {
    "Below 2.50" : 1.0,
    "2.50 - 2.99" : 0.66,
    "3.00 - 3.39" : 0.33, 
    "3.40 - 3.79" : 0.0,
    "3.80 - 4.00" : 0.0,
    "Other" : 0.33
}

In [22]:
#Application du normalisation, on le met dans un variable
S_acad = df["Current_CGPA"].map(cgpa_map)

In [23]:
#Pilier Santé Mentale
df["Depression Label"].value_counts()

Depression Label
Moderately Severe Depression    511
Severe Depression               505
Moderate Depression             457
Mild Depression                 414
Minimal Depression               97
No Depression                    44
Name: count, dtype: int64

In [24]:
depression_map = {
    "Severe Depression": 1.0,
    "Moderately Severe Depression": 0.75,
    "Moderate Depression": 0.50,
    "Mild Depression": 0.20,
    "Minimal Depression": 0.0,
    "No Depression": 0.0
}

In [25]:
df["Anxiety Label"].value_counts()

Anxiety Label
Severe Anxiety      744
Moderate Anxiety    620
Mild Anxiety        505
Minimal Anxiety     159
Name: count, dtype: int64

In [26]:
anxiety_map = {
    "Severe Anxiety": 1.0,
    "Moderate Anxiety": 0.75,
    "Mild Anxiety": 0.5,
    "Minimal Depression": 0.0,
    "No Depression": 0.0
}

In [27]:
df["Stress Label"].value_counts()

Stress Label
Moderate Stress          1348
High Perceived Stress     565
Low Stress                115
Name: count, dtype: int64

In [28]:
stress_map = {
    "High Perceived Stress": 1.0,
    "Moderate Stress": 0.5,
    "Low Stress": 0.0,
}

In [29]:
depression_score = df["Depression Label"].map(depression_map)
stress_score = df["Stress Label"].map(stress_map)
anxiety_score = df["Anxiety Label"].map(anxiety_map)

S_mental = 0.33 * depression_score + 0.33* stress_score + 0.33 * anxiety_score
S_mental

0       0.9900
1       0.6600
2          NaN
3       0.5775
4       0.9075
         ...  
2023    0.9075
2024    0.9075
2025    0.6600
2026    0.9900
2027    0.5775
Length: 2028, dtype: float64

In [30]:
df

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ6,PHQ7,PHQ8,PHQ9,Depression Value,Depression Label,Financial_Debtor_proxy,Tuition_UpToDate_proxy,Parental_SES_proxy,Economic_Vulnerability_Index
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,3,4,3,...,2,2,3,2,20,Severe Depression,No,Yes,Cadre_intellectuel_superieur,1
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,3,4,...,2,2,2,2,19,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,...,0,0,0,0,0,No Depression,Yes,No,Employe_technicien,3
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,1,2,...,1,2,2,1,14,Moderate Depression,No,Yes,Employe_technicien,1
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,4,4,...,3,0,3,3,20,Severe Depression,No,Yes,Employe_technicien,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,18-22,Male,Dhaka University (DU),Other,Second Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,2,2,3,18,Moderately Severe Depression,No,Yes,Employe_technicien,1
2024,18-22,Female,Bangladesh Agricultural University (BAU),Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,1,1,3,19,Moderately Severe Depression,No,Yes,Employe_technicien,1
2025,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,2,1,...,2,2,2,2,15,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2
2026,23-26,Female,Dhaka University (DU),Other,Third Year or Equivalent,3.40 - 3.79,No,4,4,4,...,3,3,0,0,21,Severe Depression,No,Yes,Employe_technicien,1


In [31]:
econ_base = df["Economic_Vulnerability_Index"] / 4.0
tuition_penalty = np.where(df["Tuition_UpToDate_proxy"] == "No", 0.20, 0.0)
debt_penalty = np.where(df["Financial_Debtor_proxy"] == "Yes", 0.10, 0.0)
scholarship_bonus = np.where(df["waiver_or_scholarship"] == "Yes", -0.10, 0.0)


In [32]:
S_econ = np.clip(
    econ_base + tuition_penalty + debt_penalty + scholarship_bonus, 0.0, 1.0
)

In [33]:
df["Dropout_Risk_Score"] = (
    0.35 * S_acad + 0.35 * S_mental + 0.30 * S_econ
)
df["Dropout_Risk_Score"]

0       0.652500
1       0.496500
2            NaN
3       0.392625
4       0.623625
          ...   
2023    0.508125
2024    0.508125
2025    0.612000
2026    0.421500
2027    0.508125
Name: Dropout_Risk_Score, Length: 2028, dtype: float64

In [34]:
critical_condition = (
    (S_acad == 1.0) & ((depression_score >=0.75) | (stress_score>=0.75) | (anxiety_score>=0.75)) & (df["Tuition_UpToDate_proxy"]== "No")
)

In [35]:
df.loc[critical_condition, "Dropout_Risk_Score"] = 1.0

In [36]:
df["Target_Dropout_Binary"] = (df["Dropout_Risk_Score"] >= 0.55).astype(int)

In [37]:
conditions = [
    df["Dropout_Risk_Score"] >= 0.60,
    (df["Dropout_Risk_Score"] < 0.60) | (df["Dropout_Risk_Score"].isna()),
    
]

In [38]:
choices = ["High Risk", "Low Risk"]
df["Target_Dropout_Class"] = np.select(conditions, choices)

In [39]:
df["Target_Dropout_Class"].value_counts()

Target_Dropout_Class
Low Risk     1718
High Risk     310
Name: count, dtype: int64

In [40]:
df 

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ9,Depression Value,Depression Label,Financial_Debtor_proxy,Tuition_UpToDate_proxy,Parental_SES_proxy,Economic_Vulnerability_Index,Dropout_Risk_Score,Target_Dropout_Binary,Target_Dropout_Class
0,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,3,4,3,...,2,20,Severe Depression,No,Yes,Cadre_intellectuel_superieur,1,0.652500,1,High Risk
1,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,3,4,...,2,19,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2,0.496500,0,Low Risk
2,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,0,0,0,...,0,0,No Depression,Yes,No,Employe_technicien,3,NaN,0,Low Risk
3,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,3,1,2,...,1,14,Moderate Depression,No,Yes,Employe_technicien,1,0.392625,0,Low Risk
4,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,4,4,...,3,20,Severe Depression,No,Yes,Employe_technicien,1,0.623625,1,High Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023,18-22,Male,Dhaka University (DU),Other,Second Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,18,Moderately Severe Depression,No,Yes,Employe_technicien,1,0.508125,0,Low Risk
2024,18-22,Female,Bangladesh Agricultural University (BAU),Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,3.00 - 3.39,No,4,4,4,...,3,19,Moderately Severe Depression,No,Yes,Employe_technicien,1,0.508125,0,Low Risk
2025,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,2.50 - 2.99,No,4,2,1,...,2,15,Moderately Severe Depression,No,Yes,Ouvrier_metier_manuel,2,0.612000,1,High Risk
2026,23-26,Female,Dhaka University (DU),Other,Third Year or Equivalent,3.40 - 3.79,No,4,4,4,...,0,21,Severe Depression,No,Yes,Employe_technicien,1,0.421500,0,Low Risk


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2028 entries, 0 to 2027
Data columns (total 46 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Age                           2028 non-null   object 
 1   Gender                        2028 non-null   object 
 2   University                    2028 non-null   object 
 3   Department                    2028 non-null   object 
 4   Academic_Year                 2028 non-null   object 
 5   Current_CGPA                  2028 non-null   object 
 6   waiver_or_scholarship         2028 non-null   object 
 7   PSS1                          2028 non-null   int64  
 8   PSS2                          2028 non-null   int64  
 9   PSS3                          2028 non-null   int64  
 10  PSS4                          2028 non-null   int64  
 11  PSS5                          2028 non-null   int64  
 12  PSS6                          2028 non-null   int64  
 13  PSS

# Augmentation du ligne de donnée
Pour avoir un meilleur prediction, il faut augmenter le ligne de donnée jusqu'à minimum à 5000 ligne et aussi rééquilibrer la classe, tout ca , on le fait avec SMOTENC

In [42]:
df.isna().sum()

Age                               0
Gender                            0
University                        0
Department                        0
Academic_Year                     0
Current_CGPA                      0
waiver_or_scholarship             0
PSS1                              0
PSS2                              0
PSS3                              0
PSS4                              0
PSS5                              0
PSS6                              0
PSS7                              0
PSS8                              0
PSS9                              0
PSS10                             0
Stress Value                      0
Stress Label                      0
GAD1                              0
GAD2                              0
GAD3                              0
GAD4                              0
GAD5                              0
GAD6                              0
GAD7                              0
Anxiety Value                     0
Anxiety Label               

In [43]:
df["Dropout_Risk_Score"].fillna(df["Dropout_Risk_Score"].mean(), inplace=True)
from ctgan import CTGAN

/tmp/ipykernel_469621/609582043.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Dropout_Risk_Score"].fillna(df["Dropout_Risk_Score"].mean(), inplace=True)


In [44]:
categorical_col = df.select_dtypes(include=['object', 'category'])

In [45]:
variables_categoriel = categorical_col.columns.tolist()

In [46]:
variables_categoriel

['Age',
 'Gender',
 'University',
 'Department',
 'Academic_Year',
 'Current_CGPA',
 'waiver_or_scholarship',
 'Stress Label',
 'Anxiety Label',
 'Depression Label',
 'Financial_Debtor_proxy',
 'Tuition_UpToDate_proxy',
 'Parental_SES_proxy',
 'Target_Dropout_Class']

In [47]:
for col in variables_categoriel:
    df[col] = df[col].astype(str)

In [48]:
# Séparer les données pour garantir la qualité de génération spécifique à chaque classe
df_high = df[df['Target_Dropout_Class'] == 'High Risk'].reset_index(drop=True)

In [49]:
df_low = df[df['Target_Dropout_Class'] == 'Low Risk'].reset_index(drop=True)

In [50]:
EPOCHS = 300
BATCH_SIZE = 200

In [51]:
from ctgan import CTGAN
print("--- Entraînement CTGAN pour 'High Risk' ---")
ctgan_high = CTGAN(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True, cuda=True)
ctgan_high.fit(df_high, variables_categoriel)

--- Entraînement CTGAN pour 'High Risk' ---


/home/ricardo/anaconda3/lib/python3.12/site-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Gen. (-00.47) | Discrim. (+00.31): 100%|██████| 300/300 [00:30<00:00,  9.97it/s]


In [52]:
print("\n--- Entraînement CTGAN pour 'Low Risk' ---")
ctgan_low = CTGAN(epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=True, cuda=True)
ctgan_low.fit(df_low, variables_categoriel)

/home/ricardo/anaconda3/lib/python3.12/site-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(



--- Entraînement CTGAN pour 'Low Risk' ---


Gen. (-02.27) | Discrim. (-00.45): 100%|██████| 300/300 [04:06<00:00,  1.22it/s]


In [62]:
#Génération synthétique & Rééquilibrage
TARGET_PER_CLASS = 3000

In [63]:
needed_high = max(0, TARGET_PER_CLASS - len(df_high))
needed_low = max(0, TARGET_PER_CLASS - len(df_low))

In [64]:
# Génération ciblée
synthetic_high = ctgan_high.sample(needed_high)
synthetic_high['Target_Dropout_Class'] = 'High Risk'
synthetic_high['Target_Dropout_Binary'] = 1

synthetic_low = ctgan_low.sample(needed_low)
synthetic_low['Target_Dropout_Class'] = 'Low Risk'
synthetic_low['Target_Dropout_Binary'] = 0

In [65]:
# 4. Fusion : Données Réelles + Données Synthétiques
# ==========================================
# On concatène le dataframe original avec les nouvelles lignes générées
df_augmented = pd.concat([df, synthetic_high, synthetic_low], ignore_index=True)

# Mélange aléatoire des lignes
df_augmented = df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

In [66]:
# Ajustement/Nettoyage final
if 'Dropout_Risk_Score' in df_augmented.columns:
    df_augmented['Dropout_Risk_Score'] = df_augmented['Dropout_Risk_Score'].clip(0.0, 1.0)

In [67]:
output_path = "./data/data_cleaned/Processed_augmented_5000.csv"
df_augmented.to_csv(output_path, index=False)

In [68]:
df_augmented.isna().sum()

Age                             0
Gender                          0
University                      0
Department                      0
Academic_Year                   0
Current_CGPA                    0
waiver_or_scholarship           0
PSS1                            0
PSS2                            0
PSS3                            0
PSS4                            0
PSS5                            0
PSS6                            0
PSS7                            0
PSS8                            0
PSS9                            0
PSS10                           0
Stress Value                    0
Stress Label                    0
GAD1                            0
GAD2                            0
GAD3                            0
GAD4                            0
GAD5                            0
GAD6                            0
GAD7                            0
Anxiety Value                   0
Anxiety Label                   0
PHQ1                            0
PHQ2          

In [69]:
df_augmented

,Age,Gender,University,Department,Academic_Year,Current_CGPA,waiver_or_scholarship,PSS1,PSS2,PSS3,...,PHQ9,Depression Value,Depression Label,Financial_Debtor_proxy,Tuition_UpToDate_proxy,Parental_SES_proxy,Economic_Vulnerability_Index,Dropout_Risk_Score,Target_Dropout_Binary,Target_Dropout_Class
0,18-22,Male,American International University Bangladesh (...,Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.00 - 3.39,No,2,3,2,...,2,15,Moderately Severe Depression,Yes,Yes,Employe_technicien,2,0.497625,0,Low Risk
1,23-26,Male,Dhaka University (DU),Engineering - CS / CSE / CSC / Similar to CS,Fourth Year or Equivalent,2.50 - 2.99,No,2,2,4,...,0,5,Severe Depression,No,Yes,Ouvrier_metier_manuel,2,0.656082,1,High Risk
2,18-22,Male,Rajshahi University (RU),Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,3.00 - 3.39,Yes,2,4,3,...,0,15,Moderately Severe Depression,Yes,Yes,Employe_technicien,1,0.450375,0,Low Risk
3,18-22,Male,Bangladesh University of Engineering and Techn...,Engineering - CS / CSE / CSC / Similar to CS,First Year or Equivalent,Other,No,2,3,4,...,3,17,Severe Depression,No,Yes,Ouvrier_metier_manuel,0,0.653472,1,High Risk
4,18-22,Female,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Second Year or Equivalent,Below 2.50,No,0,2,2,...,0,3,Mild Depression,No,Yes,Employe_technicien,1,0.456793,0,Low Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,23-26,Male,"Independent University, Bangladesh (IUB)",Business and Entrepreneurship Studies,First Year or Equivalent,2.50 - 2.99,No,4,2,2,...,3,17,Severe Depression,No,No,Non_specifie_ou_sans_emploi,1,0.685173,1,High Risk
5996,18-22,Male,North South University (NSU),Engineering - CS / CSE / CSC / Similar to CS,Third Year or Equivalent,Other,No,2,2,2,...,0,17,Moderate Depression,Yes,Yes,Employe_technicien,1,0.266738,0,Low Risk
5997,18-22,Male,"Independent University, Bangladesh (IUB)",Engineering - CS / CSE / CSC / Similar to CS,Fourth Year or Equivalent,Other,No,3,2,2,...,0,3,Moderate Depression,No,No,Employe_technicien,2,0.462164,0,Low Risk
5998,23-26,Male,Islamic University of Technology (IUT),Engineering - EEE/ ECE / Similar to EEE,First Year or Equivalent,3.40 - 3.79,No,3,4,4,...,3,32,Severe Depression,No,Yes,Cadre_intellectuel_superieur,0,0.272603,0,Low Risk


In [70]:
df_augmented["Target_Dropout_Class"].value_counts()

Target_Dropout_Class
Low Risk     3000
High Risk    3000
Name: count, dtype: int64